# Réseau RBF - Rendu 3

- **Dataset original** : images 256×256. Les modèles reçoivent une **copie redimensionnée** en `TAILLE_IMAGE × TAILLE_IMAGE × 3` (8×8 = 192 variables ; 16×16 = 768).
- **Algorithmes** : apprentissage/prédiction en C++ via `ctypes` et `libPA_V2.dll` ; Python charge les images, exécute les expériences et produit les figures.
- **Trois classes** : stratégie *one-vs-rest* : 3 classifieurs binaires séparés, puis `argmax` des trois **scores continus** (`decision_rbf`).
- **Protocole** : échantillon initial équilibré **dans le train seulement** → train/validation → sélection hyperparamètres → réentraînement sur les images sélectionnées du train → évaluation sur test (une fois).


## 1. Configuration , modifie seulement ces valeurs pour commencer

Commencer avec **30 images d'entraînement par classe** (soit ~90 en tout), `TAILLE_IMAGE=8` et `MODE_RAPIDE=True`. Pour agrandir : `IMAGES_TRAIN_PAR_CLASSE = 100`, puis `200`, etc. `None` = totalité du train nettoyé, **coûteux**.

In [ ]:
from pathlib import Path
import os, ctypes, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import sys
sys.path.insert(0, "../python")
import images_utils as iu

# À MODIFIER : dossier racine contenant train/<classe>/ + test/<classe>/
# ou <classe>/train/ + <classe>/test/
DATASET_DIR = Path("../dataset")
DLL_PATH = None
for nom in ["libPA_V2.dylib", "libPA_V2.so", "libPA_V2.dll"]:   # compilée par CMake dans build/
    if (Path("../build") / nom).exists():
        DLL_PATH = Path("../build") / nom

TAILLE_IMAGE = 8                # 8 puis 16 ; les images 256x256 d'origine restent inchangées
IMAGES_TRAIN_PAR_CLASSE = 30   # 30 -> 100 -> 200 -> None (tout le train)
MODE_RAPIDE = True              # petite grille d'hyperparamètres pour valider le pipeline
NETTOYAGE_DOUBLONS = True       # dédoublonnage simple + miroirs (sans détection des rotations)
FRACTION_VALIDATION = 0.20
SEED = 42
RESULTS_DIR = Path('../results/RBF_dataset')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert DLL_PATH is not None, "Bibliothèque introuvable : compile le projet avec CMake (dossier build/)."
assert iu.organisation(str(DATASET_DIR)) is not None, (
    f"Dataset introuvable ou organisation incorrecte : {DATASET_DIR}. Modifier DATASET_DIR."
)
print('Dataset:', DATASET_DIR)
print('DLL:', DLL_PATH)
print('Taille:', TAILLE_IMAGE, 'x', TAILLE_IMAGE, '=>', 3*TAILLE_IMAGE**2, 'variables')

## 2. Chargement, nettoyage et sous-échantillonnage

L'ordre alphabétique de `CLASSES` sert à coder les classes. Aucune image du **test** n'est utilisée pour choisir les hyperparamètres. Le nettoyage se fait à **32×32**, puis on recharge les mêmes indices à la taille `TAILLE_IMAGE`. Pour démarrer, le nettoyage par défaut `iu.nettoyer_train_test` peut prendre du temps sur plusieurs milliers d'images. Tu peux le désactiver pour un diagnostic technique, mais pas présenter ce test non nettoyé comme évaluation stricte.

In [ ]:
CLASSES = iu.lister_classes_dataset(str(DATASET_DIR))
NB_CLASSES = len(CLASSES)
assert NB_CLASSES == 3, f"Trois classes attendues, trouvé : {CLASSES}"
print('Classes (ordre officiel):', CLASSES)

X32_tr, y_tr_all, chemins_tr, _ = iu.charger_dataset(str(DATASET_DIR), 'train', CLASSES, 32)
X32_te, y_te_all, chemins_te, _ = iu.charger_dataset(str(DATASET_DIR), 'test', CLASSES, 32)
print('Effectifs train bruts:', np.bincount(y_tr_all, minlength=NB_CLASSES))
print('Effectifs test bruts :', np.bincount(y_te_all, minlength=NB_CLASSES))

if NETTOYAGE_DOUBLONS:
    idx_tr_all, idx_te = iu.nettoyer_train_test(X32_tr, X32_te, miroirs=True)
else:
    idx_tr_all = np.arange(len(y_tr_all))
    idx_te = np.arange(len(y_te_all))

# Échantillonnage stratifié : on ne prélève QUE dans le train nettoyé.
rng = np.random.default_rng(SEED)
chosen = []
for c in range(NB_CLASSES):
    indices = idx_tr_all[y_tr_all[idx_tr_all] == c]
    if len(indices) < 3:
        raise ValueError(f'Pas assez d’images train propres pour {CLASSES[c]}: {len(indices)}')
    rng.shuffle(indices)
    chosen.extend(indices[:len(indices) if IMAGES_TRAIN_PAR_CLASSE is None
                          else min(IMAGES_TRAIN_PAR_CLASSE, len(indices))])
idx_tr = np.sort(np.asarray(chosen, dtype=int))

def charger_taille(t):
    if t == 32:
        return X32_tr, X32_te
    A, _, _, _ = iu.charger_dataset(str(DATASET_DIR), 'train', CLASSES, t)
    B, _, _, _ = iu.charger_dataset(str(DATASET_DIR), 'test', CLASSES, t)
    return A, B

img_train, img_test = charger_taille(TAILLE_IMAGE)
X_train = np.ascontiguousarray(iu.vers_vecteurs(img_train[idx_tr]), dtype=np.float64)
X_test = np.ascontiguousarray(iu.vers_vecteurs(img_test[idx_te]), dtype=np.float64)
y_train = y_tr_all[idx_tr]
y_test = y_te_all[idx_te]
Y_train_ovr = np.ascontiguousarray(iu.etiquettes_pm1(y_train, NB_CLASSES), dtype=np.float64)

# Validation stratifiée à partir du train SEULEMENT.
idx_fit, idx_val = iu.separer_validation(y_train, fraction=FRACTION_VALIDATION, graine=SEED)
assert len(idx_fit) > 0 and len(idx_val) > 0
assert set(idx_fit).isdisjoint(set(idx_val))
print('Train choisi:', X_train.shape, np.bincount(y_train, minlength=NB_CLASSES))
print('Fit/Val:', len(idx_fit), '/', len(idx_val))
print('Test propre:', X_test.shape, np.bincount(y_test, minlength=NB_CLASSES))
print('NB: copies tournées non recherchées dans ce nettoyage simplifié. Pour une comparaison MLP stricte, réutiliser ses indices stricts.')

del X32_tr, X32_te, img_train, img_test

## 3. Interface C++ (`ctypes`) — sans sklearn et sans réentraîner à la prédiction

Les classes Python ci-dessous appellent les fonctions C++ du réseau RBF. Les modèles sont détruits à la fermeture. `status=0` signifie que l'entraînement a réussi ; les autres retours sont traités comme des erreurs.

In [ ]:
lib = ctypes.CDLL(str(DLL_PATH.resolve()))
dbl_p = ctypes.POINTER(ctypes.c_double)
vptr = ctypes.c_void_p
i32 = ctypes.c_int32
f64 = ctypes.c_double

def ptr(a):
    return a.ctypes.data_as(dbl_p)

def arr2(X):
    X = np.ascontiguousarray(X, dtype=np.float64)
    if X.ndim == 1: X = X.reshape(1, -1)
    if X.ndim != 2: raise ValueError('X doit être de forme (n, d)')
    return X

# ---- RBF network ----
lib.create_rbf_model.argtypes = [i32, i32, f64]
lib.create_rbf_model.restype = vptr
lib.train_rbf_model.argtypes = [vptr, dbl_p, dbl_p, i32, i32, f64, i32]
lib.train_rbf_model.restype = i32
lib.decision_rbf.argtypes = [vptr, dbl_p]
lib.decision_rbf.restype = f64
lib.get_num_rbf_centers.argtypes = [vptr]
lib.get_num_rbf_centers.restype = i32
lib.destroy_rbf_model.argtypes = [vptr]
lib.destroy_rbf_model.restype = None

class BinaryCppModel:
    def __init__(self, n_features, **params):
        self.d, self.params = n_features, params
        self.h = lib.create_rbf_model(n_features, int(params['K']), float(params['gamma']))
        if not self.h: raise RuntimeError('Échec création modèle RBF')

    def fit(self, X, y):
        X = arr2(X)
        y = np.ascontiguousarray(y, dtype=np.float64).reshape(-1)
        if len(X) != len(y) or X.shape[1] != self.d:
            raise ValueError('Dimensions de X/Y incompatibles')
        if not {-1.0, 1.0}.issubset(set(np.unique(y))):
            raise ValueError('Chaque RBF binaire doit voir des +1 ET -1')
        code = lib.train_rbf_model(self.h, ptr(X), ptr(y), len(X), self.d,
                                   float(self.params['lam']), int(self.params.get('maxIterations', 80)))
        if code != 0:
            raise RuntimeError(f'RBF échec de résolution: statut={code}; essaie lambda=1e-3 ou moins de centres')
        return self

    def decision(self, X):
        X = arr2(X)
        if X.shape[1] != self.d: raise ValueError('Dimensions inattendues')
        scores = np.array([lib.decision_rbf(self.h, ptr(row)) for row in X], dtype=np.float64)
        if not np.isfinite(scores).all(): raise RuntimeError('Scores non finis, vérifier entraînement et paramètres')
        return scores

    def close(self):
        if getattr(self, 'h', None):
            lib.destroy_rbf_model(self.h)
            self.h = None
    def __del__(self):
        try: self.close()
        except Exception: pass

class MultiClasseCpp:
    def __init__(self, **params):
        self.params, self.models = params, []
    def fit(self, X, y):
        self.close()
        X = arr2(X)
        y = np.asarray(y, dtype=int)
        self.n_classes = NB_CLASSES
        try:
            for c in range(self.n_classes):
                m = BinaryCppModel(X.shape[1], **self.params)
                self.models.append(m)
                m.fit(X, np.where(y == c, 1.0, -1.0))
        except Exception:
            self.close()
            raise
        return self
    def scores(self, X):
        if len(self.models) != self.n_classes: raise RuntimeError('Modèle non entraîné')
        return np.column_stack([m.decision(X) for m in self.models])
    def predict(self, X):
        return np.argmax(self.scores(X), axis=1)
    def close(self):
        for m in self.models: m.close()
        self.models = []
    def __del__(self):
        try: self.close()
        except Exception: pass

print('Interface RBF chargée.')

## 4. Contrôle rapide avant les grilles — une seule configuration par modèle

Ici, on vérifie que **le C++ entraîne vraiment** les modèles sur le dataset réel et que les trois scores donnent des classes. Ce n'est pas encore l'évaluation finale.

In [ ]:
CONFIG_TEST_RBF = dict(K=10, gamma=0.01, lam=1e-3, maxIterations=80)

m = MultiClasseCpp(**CONFIG_TEST_RBF)
t0 = time.perf_counter()
try:
    m.fit(X_train[idx_fit], y_train[idx_fit])
    pred = m.predict(X_train[idx_val])
    print('RBF val_accuracy=', round(float(np.mean(pred == y_train[idx_val])), 4),
          'temps=', round(time.perf_counter()-t0, 2), 's')
finally:
    m.close()

## 5. Sélection des hyperparamètres sur VALIDATION

Grilles volontairement courtes quand `MODE_RAPIDE=True`. Pour une expérience de recherche plus large, passer `MODE_RAPIDE=False`, mais attention au temps d'entraînement. Les scores de **test** ne sont pas utilisés ici.

Attention pour les images : `gamma` dépend fortement du nombre de dimensions (`TAILLE_IMAGE`) et de l'échelle des pixels. Il faut donc le rechercher plutôt que supposer que `gamma=1` est optimal.

In [ ]:
if MODE_RAPIDE:
    rbf_grid = [dict(K=k, gamma=g, lam=1e-3, maxIterations=80)
                for k in (5, 10, 20) for g in (0.001, 0.01)]
else:
    rbf_grid = [dict(K=k, gamma=g, lam=lam, maxIterations=100)
                for k in (5, 10, 20, 40) for g in (0.0001, 0.001, 0.01, 0.1)
                for lam in (1e-3, 1e-2)]
# RBF refuse K>N : adapter automatiquement après sous-échantillonnage.
rbf_grid = [p for p in rbf_grid if p['K'] <= len(idx_fit)]

def essayer_grille(grid):
    results = []
    for j, p in enumerate(grid, 1):
        t0 = time.perf_counter()
        m = MultiClasseCpp(**p)
        try:
            m.fit(X_train[idx_fit], y_train[idx_fit])
            acc_tr = float(np.mean(m.predict(X_train[idx_fit]) == y_train[idx_fit]))
            acc_val = float(np.mean(m.predict(X_train[idx_val]) == y_train[idx_val]))
            record = dict(params=p, train_acc=acc_tr, val_acc=acc_val,
                          secondes=time.perf_counter()-t0, error=None)
            print(f'RBF {j}/{len(grid)} : val={acc_val:.2%}, train={acc_tr:.2%} | {p}')
        except Exception as e:
            record = dict(params=p, train_acc=np.nan, val_acc=np.nan,
                          secondes=time.perf_counter()-t0, error=str(e))
            print(f'ECHEC RBF {j}/{len(grid)} : {p} -> {e}')
        finally:
            m.close()
        results.append(record)
    ok = [r for r in results if r['error'] is None]
    if not ok: raise RuntimeError('Aucune configuration RBF n’a abouti')
    best = sorted(ok, key=lambda r: (-r['val_acc'], r['secondes']))[0]
    print(f'\nMeilleur RBF : validation {best["val_acc"]:.2%}, paramètres {best["params"]}\n')
    return results, best

res_rbf, best_rbf = essayer_grille(rbf_grid)

## 6. Figures des expérimentations — comparaison des noyaux, C, gamma, K

Les points montrent des **accuracies de validation** : on évite d'utiliser le jeu de test pour choisir les paramètres.

In [ ]:
def tracer_scores(resultats):
    valid = [r for r in resultats if r['error'] is None]
    labels = [f'K={r["params"]["K"]}\ng={r["params"]["gamma"]}\nλ={r["params"]["lam"]}' for r in valid]
    fig, ax = plt.subplots(figsize=(max(9, len(labels)*0.9), 5))
    ix = np.arange(len(valid))
    ax.bar(ix-0.18, [r['train_acc'] for r in valid], 0.36, label='Train')
    ax.bar(ix+0.18, [r['val_acc'] for r in valid], 0.36, label='Validation')
    ax.set_xticks(ix, labels, fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Accuracy')
    ax.set_title('RBF — sensibilité aux hyperparamètres (échantillon train)')
    ax.legend(); ax.grid(axis='y', alpha=0.2)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'validation_rbf.png', dpi=150)
    plt.show()

tracer_scores(res_rbf)

fig, ax = plt.subplots(figsize=(5.5,4))
ok = [r for r in res_rbf if r['error'] is None]
ax.scatter([r['secondes'] for r in ok], [r['val_acc'] for r in ok])
ax.set_xlabel('Temps entraînement (s)'); ax.set_ylabel('Accuracy validation')
ax.set_title('RBF : coût vs performance'); ax.grid(alpha=0.2)
fig.tight_layout(); fig.savefig(RESULTS_DIR / 'temps_vs_validation.png', dpi=150); plt.show()

## 7. Modèles finaux — réentraînement sur toutes les images **retenues du train**, puis test

Après sélection via la validation, on réentraîne avec l'ensemble `X_train` sélectionné. Le test est ensuite évalué sans aucun ajustement supplémentaire. Pour une évaluation finale sur tout le train nettoyé, remettre `IMAGES_TRAIN_PAR_CLASSE=None` et relancer **depuis le début** . Ne pas choisir les hyperparamètres en regardant le score test.

In [ ]:
final_rbf = MultiClasseCpp(**best_rbf['params'])

try:
    t0=time.perf_counter(); final_rbf.fit(X_train, y_train); temps_final_rbf=time.perf_counter()-t0
    pred_rbf = final_rbf.predict(X_test)
    print('RBF TEST :', f'{np.mean(pred_rbf==y_test):.2%}',
          '| entraînement:', round(temps_final_rbf,2),'s')
except Exception:
    final_rbf.close()
    raise

## 8. Matrices de confusion, précision/rappel/F1 et erreurs sur images

Aucune bibliothèque `sklearn` requise : les métriques sont calculées directement. Chaque ligne d'une matrice = classe réelle ; chaque colonne = classe prédite.

In [ ]:
def matrice_confusion(y_vrai, y_pred, k):
    M = np.zeros((k,k), dtype=int)
    for a,b in zip(y_vrai,y_pred): M[int(a),int(b)] += 1
    return M

def metriques(M):
    tp = np.diag(M).astype(float)
    precision = np.divide(tp, M.sum(0), out=np.zeros(len(M)), where=M.sum(0)!=0)
    recall = np.divide(tp, M.sum(1), out=np.zeros(len(M)), where=M.sum(1)!=0)
    f1 = np.divide(2*precision*recall, precision+recall,
                   out=np.zeros(len(M)), where=(precision+recall)!=0)
    return dict(accuracy=float(tp.sum()/M.sum()), macro_f1=float(f1.mean()),
                precision=precision, recall=recall, f1=f1)

def dessiner_confusion(ax,M,titre):
    ax.imshow(M,cmap='Blues')
    ax.set_xticks(range(NB_CLASSES),CLASSES,rotation=35,ha='right')
    ax.set_yticks(range(NB_CLASSES),CLASSES)
    ax.set_xlabel('Classe prédite'); ax.set_ylabel('Vraie classe'); ax.set_title(titre)
    for i in range(NB_CLASSES):
        for j in range(NB_CLASSES):
            ax.text(j,i,str(M[i,j]),ha='center',va='center',
                    color='white' if M[i,j]>M.max()/2 else 'black')

M_rbf = matrice_confusion(y_test,pred_rbf,NB_CLASSES)
perf_rbf = metriques(M_rbf)
fig,ax=plt.subplots(figsize=(6,5))
dessiner_confusion(ax,M_rbf,'RBF réseau — test propre')
fig.tight_layout(); fig.savefig(RESULTS_DIR/'matrice_confusion.png',dpi=150); plt.show()

print(f'\nRBF: accuracy={perf_rbf["accuracy"]:.2%}  macro-F1={perf_rbf["macro_f1"]:.2%}')
for i,c in enumerate(CLASSES):
    print(f'  {c:15} précision={perf_rbf["precision"][i]:.2%} rappel={perf_rbf["recall"][i]:.2%} F1={perf_rbf["f1"][i]:.2%}')

fig,ax=plt.subplots(figsize=(6,4))
ax.bar(np.arange(NB_CLASSES),perf_rbf['f1'],.5,label='RBF')
ax.set_xticks(np.arange(NB_CLASSES),CLASSES); ax.set_ylim(0,1.05)
ax.set_ylabel('F1 par classe (test)'); ax.legend(); fig.tight_layout()
fig.savefig(RESULTS_DIR/'f1_par_classe.png',dpi=150); plt.show()

In [ ]:
def montrer_erreurs(pred, nom, nb=9):
    incorrects=np.where(pred!=y_test)[0]
    print(f'{nom} : {len(incorrects)} erreurs / {len(y_test)} images test')
    if not len(incorrects): return
    selection=incorrects[:nb]
    colonnes=3; lignes=int(np.ceil(len(selection)/colonnes))
    fig,axes=plt.subplots(lignes,colonnes,figsize=(10,3.5*lignes),squeeze=False)
    for a in axes.flat: a.axis('off')
    for ax,i in zip(axes.flat,selection):
        im=Image.open(chemins_te[idx_te[i]]).convert('RGB')
        ax.imshow(im); ax.set_title(f'Vrai : {CLASSES[y_test[i]]}\nPrédit : {CLASSES[pred[i]]}',fontsize=10)
    fig.suptitle(f'{nom} — exemples mal classés')
    fig.tight_layout(); fig.savefig(RESULTS_DIR/f'erreurs_{nom.lower()}.png',dpi=150)
    plt.show()

montrer_erreurs(pred_rbf,'RBF')

## 9. Tableau récapitulatif et export des résultats

Tous les scores affichés ci-dessous proviennent du **même test sélectionné**. Si `IMAGES_TRAIN_PAR_CLASSE` n'est pas `None`, il faut l'indiquer clairement dans ton rapport : **c'est une expérience sur sous-échantillon**, pas un modèle entraîné sur les ~2600 images.

In [ ]:
import csv, json
rows=[
    dict(model='RBF_network',n_train=len(X_train),n_test=len(X_test),taille=TAILLE_IMAGE,
         accuracy=perf_rbf['accuracy'],macro_f1=perf_rbf['macro_f1'],
         train_seconds=temps_final_rbf,params=str(best_rbf['params'])),
]
for row in rows: print(row)
with open(RESULTS_DIR/'resultats.csv','w',newline='',encoding='utf-8-sig') as f:
    writer=csv.DictWriter(f,fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
with open(RESULTS_DIR/'configuration.json','w',encoding='utf-8') as f:
    json.dump(dict(classes=CLASSES,taille=TAILLE_IMAGE,n_train=len(X_train),n_test=len(X_test),
                   train_par_classe=IMAGES_TRAIN_PAR_CLASSE,nettoyage_doublons=NETTOYAGE_DOUBLONS,
                   best_rbf=best_rbf['params']),f,indent=2,ensure_ascii=False)
print('Figures et CSV enregistrés dans',RESULTS_DIR.resolve())

## 10. Ressources et suites

- **Pour les tests rapides** : 30 images/classe, 8×8, grille courte.
- **Pour les premiers résultats comparatifs** : 100 images/classe, puis 200/classe ; conserver la séparation validation/test.
- **Pour le dataset complet** : mesurer le temps d'entraînement avant de lancer toutes les images.
- **Comparaison équitable MLP** : utiliser exactement la même sélection nettoyée et les mêmes images de test. Le pipeline MLP de ta collègue applique aussi une détection plus coûteuse de copies tournées, que ce notebook ne reproduit pas automatiquement.
- **Application** : les centres et poids du RBF sont exportés dans `models/RBF/rbf.npz` par le notebook `RBF_Fleurs_MINIMAL` ; l'API (`app/serveur.py`) les charge pour prédire.

À la fin de la session, libérer les modèles :

In [ ]:
# À exécuter APRÈS avoir fini toutes les prédictions / analyses.
final_rbf.close()
print('Mémoire C++ libérée.')